In [1]:
import os, json, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
 
BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
DATA_DIR    = os.path.join(OUT, 'data')
SCALER_DIR  = os.path.join(OUT, 'scalers')
MODEL_DIR   = os.path.join(OUT, 'models')
 
DASHBOARD_DIR = os.path.join(V2_DIR, 'dashboard_export')
os.makedirs(DASHBOARD_DIR, exist_ok=True)
 
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 
CPCB_THRESHOLD = 60.0   # µg/m³, 24h PM2.5 standard
WHO_THRESHOLD  = 15.0   # µg/m³, 24h PM2.5 guideline
 
print(f"Device: {DEVICE}")
print(f"Dashboard export will save to: {DASHBOARD_DIR}")

Device: cuda
Dashboard export will save to: D:\FYP\Eco-Twin\v2_clean\dashboard_export


In [4]:
X_test  = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
Y_test  = np.load(os.path.join(DATA_DIR, 'Y_test.npy'))
At_test = np.load(os.path.join(DATA_DIR, 'At_test.npy'))
 
with open(os.path.join(SCALER_DIR, 'pm25_scaler.pkl'), 'rb') as f:
    pm25_scaler = pickle.load(f)
with open(os.path.join(SCALER_DIR, 'feature_scaler.pkl'), 'rb') as f:
    feat_scaler = pickle.load(f)
with open(os.path.join(SCALER_DIR, 'log_scaler.pkl'), 'rb') as f:
    log_scaler = pickle.load(f)
 
with open(os.path.join(MODEL_DIR, 'final_config.json'), 'r') as f:
    config = json.load(f)
 
df_stations = pd.read_excel(os.path.join(V2_DIR, 'Delhi_list_v2.xlsx'))
df_stations.columns = df_stations.columns.str.strip().str.lower()
df_stations['station'] = df_stations['station'].str.strip()
station_names = df_stations['station'].tolist()
N = len(station_names)
 
INPUT_WINDOW, HORIZON = config['input_window'], config['forecast_horizon']
 
print(f"Test samples : {X_test.shape[0]:,}")
print(f"Stations     : {N}")
print(f"Config       : {config}")

Test samples : 2,918
Stations     : 37
Config       : {'in_features': 14, 'hidden_dim': 32, 'gru_layers': 1, 'attn_heads': 4, 'forecast_horizon': 24, 'n_stations': 37, 'dropout': 0.2, 'input_window': 48, 'log_target': True, 'seeds': [42, 7, 123, 2024, 999]}


In [5]:
_df = pd.read_csv(os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv'),
                  parse_dates=['Timestamp'], usecols=['Timestamp'])
all_timestamps = np.array(sorted(_df['Timestamp'].unique()))
T = len(all_timestamps)
 
train_end_idx, val_end_idx = int(T*0.70), int(T*0.85)
VAL_END = pd.Timestamp(all_timestamps[val_end_idx-1])
test_mask = all_timestamps > VAL_END
test_timestamps = all_timestamps[test_mask]
 
def future_time_windows(ts_subset):
    ts = pd.DatetimeIndex(ts_subset)
    tf = np.stack([ts.hour, ts.dayofweek, ts.month,
                   (ts.dayofweek >= 5).astype(int)], axis=1).astype(np.float32)
    n = len(ts) - (INPUT_WINDOW + HORIZON) + 1
    out = np.zeros((n, HORIZON, 4), dtype=np.float32)
    for i in range(n):
        out[i] = tf[i+INPUT_WINDOW : i+INPUT_WINDOW+HORIZON]
    return out
 
TF_test = future_time_windows(test_timestamps)
tm, ts_ = feat_scaler.mean_[10:14], feat_scaler.scale_[10:14]
TF_test = ((TF_test - tm) / ts_).astype(np.float32)
 
# the timestamp marking "now" (end of input window) for each test sample
window_start_ts = test_timestamps[:X_test.shape[0]]
window_now_ts   = test_timestamps[INPUT_WINDOW : INPUT_WINDOW + X_test.shape[0]]
 
print(f"Future time windows: {TF_test.shape}")
print(f"Test window 'now' timestamps range: {window_now_ts[0]} -> {window_now_ts[-1]}")

Future time windows: (2918, 24, 4)
Test window 'now' timestamps range: 2025-12-08 12:00:00 -> 2026-04-09 01:00:00


In [6]:
class WATGCN(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads,
                 forecast_horizon, n_stations, dropout=0.2, n_time_feats=4):
        super().__init__()
        self.input_proj = nn.Linear(in_features, hidden_dim)
        self.gcn_linear = nn.Linear(hidden_dim, hidden_dim)
        self.gcn_norm   = nn.LayerNorm(hidden_dim)
        self.act        = nn.ReLU()
        self.gru        = nn.GRU(hidden_dim, hidden_dim, gru_layers, batch_first=True,
                                 dropout=dropout if gru_layers > 1 else 0)
        self.attention  = nn.MultiheadAttention(hidden_dim, attn_heads,
                                                dropout=dropout, batch_first=True)
        self.attn_norm  = nn.LayerNorm(hidden_dim)
        self.dropout    = nn.Dropout(dropout)
        self.time_enc = nn.Sequential(
            nn.Linear(n_time_feats, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim))
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim*2, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH = hidden_dim, forecast_horizon
 
    def forward(self, X, At, TFut, Anchor):
        B, T, N, _ = X.shape
        H = self.act(self.input_proj(X))
        I = torch.eye(N, device=At.device).unsqueeze(0).unsqueeze(0)
        A = At + I
        A = A / A.sum(dim=-1, keepdim=True).clamp(min=1e-6)
        Hg = self.act(self.gcn_norm(self.gcn_linear(torch.matmul(A, H)))) + H
        Hf = Hg.permute(0,2,1,3).reshape(B*N, T, self.H)
        g, _ = self.gru(Hf)
        g = self.dropout(g)
        a, _ = self.attention(g, g, g)
        ctx = self.attn_norm(a + g)[:, -1, :]
        t_emb = self.time_enc(TFut).unsqueeze(1).expand(B, N, self.FH, self.H)
        t_emb = t_emb.reshape(B*N, self.FH, self.H)
        ctx_r = ctx.unsqueeze(1).expand(-1, self.FH, -1)
        resid = self.decoder(torch.cat([ctx_r, t_emb], dim=-1)).squeeze(-1)
        resid = resid.reshape(B, N, self.FH).permute(0,2,1)
        anchor = Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)
        return resid + anchor
 
def to_log(a):
    return log_scaler.transform(
        np.log1p(np.clip(a, 0, None)).reshape(-1,1)).reshape(a.shape).astype(np.float32)
 
def from_log(a):
    lg = log_scaler.inverse_transform(a.reshape(-1,1)).reshape(a.shape)
    return np.expm1(lg)
 
fm0, fs0 = feat_scaler.mean_[0], feat_scaler.scale_[0]
raw_last_pm25 = X_test[:, -1, :, 0] * fs0 + fm0     # (n_test, N) real µg/m³
Anchor_test = to_log(raw_last_pm25)
 
inv_pm = lambda a: pm25_scaler.inverse_transform(a.reshape(-1,1)).reshape(a.shape)
Yr_test = inv_pm(Y_test)
 
print("Model class + helpers ready.")
print(f"raw_last_pm25 range: [{raw_last_pm25.min():.1f}, {raw_last_pm25.max():.1f}]")

Model class + helpers ready.
raw_last_pm25 range: [0.0, 983.0]


In [7]:
models = []
for seed in config['seeds']:
    m = WATGCN(config['in_features'], config['hidden_dim'], config['gru_layers'],
              config['attn_heads'], config['forecast_horizon'], config['n_stations'],
              config['dropout']).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_final_seed{seed}.pt'),
                                 map_location=DEVICE))
    m.eval()
    models.append(m)
 
print(f"Loaded {len(models)} ensemble members.")
 
X_t  = torch.from_numpy(X_test).float()
At_t = torch.from_numpy(At_test).float()
TF_t = torch.from_numpy(TF_test).float()
An_t = torch.from_numpy(Anchor_test).float()
 
BATCH = 32
n = X_t.shape[0]
all_preds_log = []
 
with torch.no_grad():
    for i in range(0, n, BATCH):
        xb = X_t[i:i+BATCH].to(DEVICE)
        ab = At_t[i:i+BATCH].to(DEVICE)
        tb = TF_t[i:i+BATCH].to(DEVICE)
        anb = An_t[i:i+BATCH].to(DEVICE)
        batch_preds = torch.stack([m(xb, ab, tb, anb) for m in models], dim=0)
        ens = batch_preds.mean(dim=0).cpu().numpy()
        all_preds_log.append(ens)
        if (i // BATCH) % 20 == 0:
            print(f"  {i+len(xb)}/{n}")
 
preds_log = np.concatenate(all_preds_log, axis=0)
preds_real = from_log(preds_log)   # (n_test, 24, N) real µg/m³
 
print(f"\nInference complete. preds_real shape: {preds_real.shape}")
print(f"Range: [{preds_real.min():.1f}, {preds_real.max():.1f}] µg/m³")

Loaded 5 ensemble members.
  32/2918
  672/2918
  1312/2918
  1952/2918
  2592/2918

Inference complete. preds_real shape: (2918, 24, 37)
Range: [3.2, 869.5] µg/m³


In [8]:
stations_export = []
for i, name in enumerate(station_names):
    stations_export.append({
        'id': i,
        'name': name,
        'lat': float(df_stations.iloc[i]['latitude']),
        'lon': float(df_stations.iloc[i]['longitude']),
    })
 
with open(os.path.join(DASHBOARD_DIR, 'stations.json'), 'w') as f:
    json.dump(stations_export, f, indent=2)
 
print(f"Saved {len(stations_export)} stations -> stations.json")

Saved 37 stations -> stations.json


In [9]:
N_PLAYBACK = 300   # evenly spaced windows across the test set
sample_idx = np.linspace(0, n - 1, N_PLAYBACK).astype(int)
 
def aqi_category(pm25):
    if pm25 <= 30:   return 'Good'
    if pm25 <= 60:   return 'Satisfactory'
    if pm25 <= 90:   return 'Moderate'
    if pm25 <= 120:  return 'Poor'
    if pm25 <= 250:  return 'Very Poor'
    return 'Severe'
 
playback = []
for idx in sample_idx:
    now_ts = pd.Timestamp(window_now_ts[idx])
    entry = {
        'window_idx': int(idx),
        'timestamp': now_ts.strftime('%Y-%m-%d %H:%M'),
        'stations': []
    }
    for s in range(N):
        actual_24h    = Yr_test[idx, :, s].round(1).tolist()
        predicted_24h = preds_real[idx, :, s].round(1).tolist()
        current_pm25  = float(raw_last_pm25[idx, s])
        entry['stations'].append({
            'station_id': s,
            'current_pm25': round(current_pm25, 1),
            'current_aqi': aqi_category(current_pm25),
            'actual_24h': actual_24h,
            'predicted_24h': predicted_24h,
            'peak_predicted': round(max(predicted_24h), 1),
            'hours_to_breach_cpcb': next(
                (h+1 for h, v in enumerate(predicted_24h) if v > CPCB_THRESHOLD), None),
        })
    playback.append(entry)
 
with open(os.path.join(DASHBOARD_DIR, 'playback.json'), 'w') as f:
    json.dump(playback, f)
 
size_mb = os.path.getsize(os.path.join(DASHBOARD_DIR, 'playback.json')) / 1e6
print(f"Saved {len(playback)} playback windows -> playback.json ({size_mb:.1f} MB)")

Saved 300 playback windows -> playback.json (8.9 MB)


In [11]:
with open(os.path.join(V2_DIR, 'ecotwin_outputs', 'results', 'final_metrics.json'), 'r') as f:
    final_metrics = json.load(f)
 
summary = {
    'model_name': 'WA-TGCN (5-seed ensemble)',
    'n_stations': N,
    'test_windows_total': int(n),
    'test_windows_exported': len(playback),
    'test_period': f"{pd.Timestamp(window_now_ts[0]).strftime('%Y-%m-%d')} to "
                   f"{pd.Timestamp(window_now_ts[-1]).strftime('%Y-%m-%d')}",
    'rmse': final_metrics['rmse'],
    'mae': final_metrics['mae'],
    'r2': final_metrics['r2'],
    'rmse_by_horizon': final_metrics['rmse_by_horizon'],
    'rmse_by_regime': final_metrics['rmse_by_regime'],
    'cpcb_threshold': CPCB_THRESHOLD,
    'who_threshold': WHO_THRESHOLD,
}
with open(os.path.join(DASHBOARD_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
 
print("Saved summary.json")
print()
print("="*55)
print("  EXPORT COMPLETE")
print("="*55)
print(f"  {DASHBOARD_DIR}\\stations.json")
print(f"  {DASHBOARD_DIR}\\playback.json")
print(f"  {DASHBOARD_DIR}\\summary.json")
print()
print("="*55)

Saved summary.json

  EXPORT COMPLETE
  D:\FYP\Eco-Twin\v2_clean\dashboard_export\stations.json
  D:\FYP\Eco-Twin\v2_clean\dashboard_export\playback.json
  D:\FYP\Eco-Twin\v2_clean\dashboard_export\summary.json

